In [39]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from typing import Optional, List, Dict, Any
load_dotenv()

True

In [54]:
class LLMAdapter:
    def __init__(self, api_key: str,  base_url: str , model: str, timeout: int = 30):
        self.api_key = api_key
        self.model = model
        self.base_url = base_url
        self.timeout = timeout
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key)


    def __repr__(self):
        key = (
            self.api_key[:8] + "..." if len(self.api_key) > 8 else "***"
        )
        return (
            f"OpenAILLM(\n"
            f"  model='{self.model}',\n"
            f"  base_url='{self.base_url}',\n"
            f"  timeout={self.timeout},\n"
            f"  api_key='{key}'\n"
            f")"
        )
    
    def chat(self, messages: List[dict], temperature: float = 0.7, stream: bool = False):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            stream=stream,
            timeout=self.timeout
        )
        return response.choices[0].message.content

In [55]:
SYSTEM_PROMPT = """
你是一名能够调用工具解决问题的 AI Agent。

## 你的目标

你需要通过「思考 → 调用工具 → 观察结果 → 继续思考」的方式解决用户问题。

你可以使用以下工具：

{tool_desc}

---

## 输出格式

当需要调用工具时，严格按照下面格式输出：

Thought: 说明你当前的思考

Action: 工具名称

Action Input: 工具输入

---

工具返回结果后，你会收到：

Observation: 工具执行结果

然后继续思考。

---

如果已经能够回答用户问题，不要再调用工具，而是直接输出：

Final Answer: 最终答案

---

注意：

1. 每次最多调用一个工具。
2. Action 必须是提供的工具名称之一。
3. Action Input 只能填写工具需要的输入内容。
4. 不要编造 Observation，Observation 一定来自工具执行结果。
5. 如果已有足够信息，请直接输出 Final Answer。
""".strip()

In [56]:
def build_messages(
    question: str,
    tool_desc: str,
    scratchpad: str = "",
) -> list[dict]:
    """
    构造发送给 LLM 的 messages。
    """
    system_prompt = SYSTEM_PROMPT.format(
        tool_desc=tool_desc
    )
    user_prompt = f"""用户问题：
        {question}
        {scratchpad}
    """
    return [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

In [57]:
class WeatherTool:
    name = "weather"
    description = "查询指定城市的天气"

    def run(self, tool_input: str) -> str:
        return f"{tool_input}：晴天，30℃"
    
class SearchTool:
    name = "search"
    description = "搜索互联网（示例）"

    def run(self, tool_input: str) -> str:
        return f"搜索结果：{tool_input} 的相关信息。"

In [58]:
class ToolRegistry:

    def __init__(self):
        self.tools = {}

    def register(self, tool):
        """注册工具"""
        self.tools[tool.name] = tool

    def get(self, name):
        """根据名称获取工具"""
        return self.tools.get(name)

    def list_tools(self):
        """返回所有工具"""
        return list(self.tools.values())

    def tool_description(self):
        """生成 Prompt 中的 Tool Description"""
        desc = []
        for tool in self.tools.values():
            desc.append(
                f"{tool.name}：{tool.description}"
            )

        return "\n".join(desc)
    

In [ ]:
import re


def parse_response(text: str) -> dict:

    text = text.strip()
    final_match = re.search(
        r"Final Answer:\s*(.*)",
        text,
        re.DOTALL,
    )

    if final_match:
        return {
            "type": "finish",
            "answer": final_match.group(1).strip(),
        }
    thought_match = re.search(
        r"Thought:\s*(.*?)\nAction:",
        text,
        re.DOTALL,
    )
    action_match = re.search(
        r"Action:\s*(.*?)\nAction Input:",
        text,
        re.DOTALL,
    )
    input_match = re.search(
        r"Action Input:\s*(.*)",
        text,
        re.DOTALL,
    )

    if not (thought_match and action_match and input_match):
        raise ValueError(
            f"无法解析模型输出：\n{text}"
        )

    return {
        "type": "action",
        "thought": thought_match.group(1).strip(),
        "action": action_match.group(1).strip(),
        "action_input": input_match.group(1).strip(),
    }

In [60]:
class ReActAgent:
    def __init__(self, llm: LLMAdapter, registry, max_steps: int = 5):
        self.llm = llm
        self.registry = registry
        self.max_steps = max_steps

    def run(self, question: str) -> str:
        scratchpad = ""
        for step in range(self.max_steps):
            messages = build_messages(
                question=question,
                tool_desc=self.registry.tool_description(),
                scratchpad=scratchpad,
            )
            response = self.llm.chat(messages)
            print(f"\n========== Step {step + 1} ==========")
            print(response)
            result = parse_response(response)
            if result["type"] == "finish":
                return result["answer"]
            tool = self.registry.get(result["action"])
            if tool is None:
                return f"未知工具：{result['action']}"
            observation = tool.run(result["action_input"])
            print(f"\nObservation: {observation}")
            scratchpad += f"""
                Thought: {result['thought']}
                Action: {result['action']}
                Action Input: {result['action_input']}
                Observation: {observation}

                """
        return "达到最大推理步数，任务终止。"


In [61]:
llm = LLMAdapter(api_key=os.getenv("API_KEY"), base_url=os.getenv("BASE_URL"), model=os.getenv("MODEL"), timeout=30)
llm

OpenAILLM(
  model='qwen3.7-max',
  base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
  timeout=30,
  api_key='sk-ws-H....'
)

In [62]:
registry = ToolRegistry()
registry.register(WeatherTool())
registry.register(SearchTool())
agent = ReActAgent(registry=registry, llm=llm, max_steps=5)
agent.run("请帮我查询北京的天气，并搜索相关信息。")


========== Step 1 ==========
Thought: 用户需要查询北京的天气并搜索相关信息。首先，我将使用 weather 工具查询北京的当前天气情况。
Action: weather
Action Input: 北京

Observation: 北京：晴天，30℃

========== Step 2 ==========
Thought: 已经获取了北京的天气情况（晴天，30℃），接下来需要使用 search 工具搜索北京的相关天气信息或近期新闻，以提供更全面的回答。
Action: search
Action Input: 北京天气

Observation: 搜索结果：北京天气 的相关信息。

========== Step 3 ==========
Final Answer: 北京的当前天气为晴天，气温30℃。同时，关于“北京天气”的相关搜索信息也已获取，您可以随时了解最新的天气动态和资讯。如果您需要更详细的天气预报或特定方面的信息，请随时告诉我！


'北京的当前天气为晴天，气温30℃。同时，关于“北京天气”的相关搜索信息也已获取，您可以随时了解最新的天气动态和资讯。如果您需要更详细的天气预报或特定方面的信息，请随时告诉我！'